# Pseudotime-Associated Genes and Trajectory Heatmaps in Malignant Cells

Correlates gene expression with Monocle3 pseudotime across the
malignant compartment, then plots smoothed expression of curated marker panels
along pseudotime for paths.

In [ ]:
# !pip install --quiet scanpy statsmodels
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/tumour/pulmonary_epi_Lin/

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.colors import ListedColormap
from scipy.ndimage import gaussian_filter1d
from scipy.stats import rankdata
from scipy.stats import t as t_dist
from statsmodels.stats.multitest import multipletests

## Config

In [ ]:
input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join_cytotrace_monocle_malignant.h5ad"
figures_output_dir = "./figures/monocle"
tables_output_dir = "./tables"

state_col = "final_state"
pseudotime_col = "monocle_pseudotime"   # Spearman is rank-based, so raw or scaled gives identical results
counts_layer = "raw_counts"
min_cells_per_gene = 10
gene_chunk_size = 500                   # genes per block in the vectorised Spearman (memory vs speed)

state_colors = {
    "1_Epithelial baseline": "#8dd3c7",
    "2_Differentiated epithelial": "#bebada",
    "3_Devp_stem-like": "#80b1d3",
    "4_Inflammatory EMT": "#fccde5",
    "5_Neurod_EMT-like": "#bc80bd",
    "6_Proliferative": "#ffed6f",
}

# trajectory paths (sets of states; cells are ordered by pseudotime within each)
paths = {
    "path1": ["1_Epithelial baseline", "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative"],
    "path2": ["1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like"],
}

# curated marker panels (heatmap rows, in display order)
markers_path1 = [
    "SFTPB", "HOPX", "NKX2-1", "NAPSA", "CEACAM6", "CEACAM5", "MUC4", "MUC21", "KRT7", "KRT8", "KRT5",
    "CXCL2", "CXCL14", "IFI44L", "LGR4", "POU2F3", "POU6F2", "EGFR", "IL33", "INPP4B",
    "SETBP1", "IGF2BP3", "IGF2BP2", "SYNE2", "HMGA2", "KIF4A", "MUC5B", "CDH11", "TP63", "LSAMP", "SOX4",
]
markers_path2_tf = [
    "SFTPB", "HOPX", "NKX2-1", "NAPSA", "CEACAM6", "CEACAM5", "KRT7", "ST6GALNAC5",
    "MAP2", "CD55", "KLF6", "WNT9A", "WIF1", "TEAD1", "LGR4", "KLF3", "TCF7", "ATF7", "KMT2A", "TCF7L2",
    "AXIN2", "LEF1", "ROR1", "ROR2", "EGFR", "CD44", "ITGA6", "SOX5", "FAT1", "PRKCA", "SETBP1", "CD36",
]

# heatmap display
smoothing_sigma = 12                    # Gaussian sigma in cells along pseudotime
pseudotime_bins = None                  # e.g. 200: average cells into equal-count bins before smoothing
heatmap_cmap = "coolwarm"
pseudotime_bar_cmap = "GnBu"
pseudotime_bar_percentiles = (5, 95)    # pseudotime bar colour limits, per heatmap

heatmaps = [
    {"file": "13_heatmap_path1.pdf", "path": "path1", "genes": markers_path1,
     "title": "Path 1 (baseline → inflammatory EMT → neural EMT-like → proliferative)", "vlim": 2, "figsize": (6, 5)},
    {"file": "14_heatmap_path2_TF.pdf", "path": "path2", "genes": markers_path2_tf,
     "title": "Path 2 (baseline → differentiated → stem-like)", "vlim": 1.5, "figsize": (6, 5)},
]

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)

## 1. Load and normalise

In [ ]:
adata = sc.read(input_path)
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
sc.pp.filter_genes(adata, min_cells=min_cells_per_gene)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.obs[state_col].value_counts()

## 2. Genes correlated with pseudotime

Spearman correlation of log-normalised expression with pseudotime, computed exactly
on all cells in blocks of genes (same statistic and t-approximation p-value as
`scipy.stats.spearmanr`). With tens of thousands of cells nearly every expressed
gene reaches padj < 0.05, so genes are ranked by |ρ|; negative ρ = early, positive = late.

In [ ]:
def pseudotime_gene_correlation(ad, pseudotime_key=pseudotime_col, chunk=gene_chunk_size):
    n = ad.n_obs
    pt_rank = rankdata(ad.obs[pseudotime_key].values)
    pt_z = (pt_rank - pt_rank.mean()) / pt_rank.std()
    rho = np.full(ad.n_vars, np.nan)
    for start in range(0, ad.n_vars, chunk):
        block = ad.X[:, start:start + chunk]
        block = block.toarray() if hasattr(block, "toarray") else np.asarray(block)
        ranks = rankdata(block, axis=0)
        sd = ranks.std(axis=0)
        ok = sd > 0
        rho[start:start + chunk][ok] = ((ranks[:, ok] - ranks[:, ok].mean(axis=0)) / sd[ok]).T @ pt_z / n
    t_stat = rho * np.sqrt((n - 2) / np.clip(1 - rho**2, 1e-12, None))
    res = pd.DataFrame({"spearman_corr": rho, "pval": 2 * t_dist.sf(np.abs(t_stat), df=n - 2)},
                       index=ad.var_names).dropna()
    res["padj"] = multipletests(res["pval"], method="fdr_bh")[1]
    res.index.name = "gene"
    return res.reindex(res["spearman_corr"].abs().sort_values(ascending=False).index)

pt_results = pseudotime_gene_correlation(adata)
pt_results.to_csv(f"{tables_output_dir}/pt_monocle_results_malignant.csv")

print(f"early (rho<0, padj<0.05): {((pt_results.spearman_corr < 0) & (pt_results.padj < 0.05)).sum()}, "
      f"late (rho>0, padj<0.05): {((pt_results.spearman_corr > 0) & (pt_results.padj < 0.05)).sum()}")
pt_results.head(20)

## 3. Pseudotime heatmaps

Cells in each path are ordered by pseudotime; each gene is z-scored across those
cells and smoothed along pseudotime (Gaussian, `smoothing_sigma` cells). Top bars:
cell state and pseudotime (colour limits at the 5th–95th percentile of that path).
Rows follow the curated panel order.

In [ ]:
def pseudotime_heatmap(ad, states, genes, title, vlim=2, figsize=(6, 5), save_path=None,
                       sigma=smoothing_sigma, n_bins=pseudotime_bins):
    sub = ad[ad.obs[state_col].isin(states)]
    order = np.argsort(sub.obs[pseudotime_col].values, kind="stable")
    genes_use = [g for g in genes if g in sub.var_names]
    missing = [g for g in genes if g not in sub.var_names]
    print(f"{title}: {sub.n_obs} cells, {len(genes_use)}/{len(genes)} genes" + (f" (missing: {missing})" if missing else ""))

    expr = sub[:, genes_use].X
    expr = (expr.toarray() if hasattr(expr, "toarray") else np.asarray(expr))[order].astype(np.float32)
    sd = expr.std(axis=0)
    expr = ((expr - expr.mean(axis=0)) / np.where(sd == 0, 1, sd)).T          # genes x cells
    states_sorted = sub.obs[state_col].astype(str).values[order]
    pt_sorted = sub.obs[pseudotime_col].values[order]

    if n_bins:
        edges = np.linspace(0, expr.shape[1], n_bins + 1).astype(int)
        expr = np.stack([expr[:, a:b].mean(axis=1) for a, b in zip(edges[:-1], edges[1:])], axis=1)
        states_sorted = np.array([pd.Series(states_sorted[a:b]).mode()[0] for a, b in zip(edges[:-1], edges[1:])])
        pt_sorted = np.array([pt_sorted[a:b].mean() for a, b in zip(edges[:-1], edges[1:])])
    if sigma:
        expr = gaussian_filter1d(expr, sigma=sigma, axis=1, mode="nearest")

    lo, hi = np.percentile(pt_sorted, pseudotime_bar_percentiles)
    pt_norm = np.clip((pt_sorted - lo) / max(hi - lo, 1e-9), 0, 1)
    present = [s for s in state_colors if s in set(states_sorted)]
    state_codes = np.array([present.index(s) for s in states_sorted])

    fig, axes = plt.subplots(3, 2, figsize=figsize, gridspec_kw={
        "height_ratios": [0.35, 0.35, 12], "width_ratios": [40, 1], "hspace": 0.06, "wspace": 0.03})
    ax_state, ax_pt, ax_hm, cax = axes[0, 0], axes[1, 0], axes[2, 0], axes[2, 1]
    axes[0, 1].axis("off"); axes[1, 1].axis("off")

    ax_state.imshow(state_codes[None, :], aspect="auto", interpolation="nearest",
                    cmap=ListedColormap([state_colors[s] for s in present]), rasterized=True)
    ax_pt.imshow(pt_norm[None, :], aspect="auto", interpolation="nearest", cmap=pseudotime_bar_cmap,
                 vmin=0, vmax=1, rasterized=True)
    for ax, label in ((ax_state, "State"), (ax_pt, "Pseudotime")):
        ax.set_xticks([]); ax.set_yticks([0], [label], fontsize=8)
        for spine in ax.spines.values():
            spine.set_visible(False)

    im = ax_hm.imshow(expr, aspect="auto", interpolation="nearest", cmap=heatmap_cmap,
                      vmin=-vlim, vmax=vlim, rasterized=True)
    ax_hm.set_yticks(np.arange(len(genes_use)), genes_use, fontsize=8, fontstyle="italic")
    ax_hm.set_xticks([])
    ax_hm.set_xlabel("Cells ordered along pseudotime", fontsize=10)
    for side in ("top", "right"):
        ax_hm.spines[side].set_visible(False)
    fig.colorbar(im, cax=cax, label="Z-score")
    ax_state.set_title(title, fontsize=11, pad=8)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    return pd.DataFrame(expr, index=genes_use)

In [ ]:
for spec in heatmaps:
    pseudotime_heatmap(
        adata, paths[spec["path"]], spec["genes"], spec["title"], vlim=spec["vlim"],
        figsize=spec["figsize"], save_path=f"{figures_output_dir}/{spec['file']}")